# Hello World pipeline

In [5]:
from kfp import dsl

@dsl.component(base_image="python:3.11")
def hello_world(name:str) -> str:
    return f"hello world\nnice to meet you {name}"

@dsl.component(base_image="python:3.11")
def print_str(hello_str:str):
    print(hello_str)

In [6]:
from kfp import dsl

@dsl.pipeline(name="hello-world")
def hello_world_pipeline(name:str):
    hello_comp = hello_world(name=name)
    print_comp = print_str(hello_str=hello_comp.output)

In [7]:
from kfp.compiler import Compiler

Compiler().compile(hello_world_pipeline, package_path="hello-world.yaml")

# Finetuning pipeline

## install kfp-kubernetes

In [9]:
%%sh
pip install -q --no-cache-dir kfp[kubernetes]

## jsonl 데이터셋에 system prompt 추가하기

In [2]:
from kfp import dsl
from kfp.dsl import Dataset

@dsl.component(base_image="python:3.11", packages_to_install=["datasets"])
def add_system_prompt_to_dataset(
    dataset_jsonl_file: str,
    system_prompt: str,
    output_dataset: Dataset,
):
    
    from datasets import load_dataset
    
    dataset = load_dataset("json", data_files=dataset_jsonl_file)

    def add_system_prompt(examples):
        """add system prompt to a sample of the dataset."""
        for messages in examples['messages']:
            messages.insert(0, {"role": "system", "content": system_prompt})
        return examples

    dataset = dataset.map(add_system_prompt, batched=True, remove_columns=dataset['train'].column_names)
    # save as jsonl file
    dataset['train'].to_json(output_dataset.path)

## training code

In [63]:
from kfp import dsl

@dsl.component(base_image="python:3.11", packages_to_install=["requests"])
def get_train_func(training_code_url:str) -> str:
    import requests

    response = requests.get(training_code_url)
    training_code_str = str(response.content)

    print(training_code_str)
    
    return training_code_str

## deepspeed config 만들기

In [59]:
from kfp import dsl

@dsl.component(base_image="python:3.11", packages_to_install=["requests"])
def save_deepspeed_config(
    deepspeed_config_url:str,
    deepspeed_config_extra:str,
) -> dict:
    import requests
    import json
    response = requests.get(deepspeed_config_url)
    deepspeed_config = json.loads(response)

    # add extra configuration
    if deepspeed_config_extra is not None:
        deepspeed_config.update(json.loads(deepspeed_config_extra))
    
    print(deepspeed_config)

    return deepspeed_config

## PytorchJob 만들기

In [ ]:
from kfp import dsl
from kfp.dsl import Dataset

@dsl.component(base_image="python:3.11", packages_to_install=["kubeflow-training"])
def run_pytorchjob(
    base_image:str,
    pvc_name:str,
    training_code:str,
    pretrained_model_name:str,
    train_dataset_name:Dataset,
    eval_dataset_name:Dataset,
    job_name:str,
    deepspeed_config:dict,
    batch_size:int,
    num_train_epochs:int,
    master_replica:int,
    master_resources:str,
    worker_replica:int,
    worker_resources:str,
) -> str:
    import string
    import random
    import os
    import json

    ## create job id
    letters_set = string.ascii_lowercase + string.digits
    random_list = random.sample(letters_set,5)
    job_id = ''.join(random_list)

    ## parse master and worker resources to dictionary
    master_resources = json.loads(master_resources)
    worker_resources = json.loads(worker_resources)
    print(f"master resources: \n{json.dumps(master_resources, indend=4)}")
    print(f"worker resources: \n{json.dumps(worker_resources, indend=4)}")

    ## deepspeed
    deepspeed_config = json.dumps(deepspeed_config)

    ## save dataset
    mount_path = "/train"
    job_workspace = os.path.join(mount_path, f"{job_name}-{job_id}")
    os.makedirs(job_workspace, exist_ok=True)
    with open(os.path.join(job_workspace, "train.jsonl"), "w") as f:
        f.write(open(train_dataset_name.path).read())
    if eval_dataset_name is not None:
        with open(os.path.join(job_workspace, "eval.jsonl"), "w") as f:
            f.write(open(eval_dataset_name.path).read())
    
    ## populate exec script
    
    
    exec_script = """
program_path=$(mktemp -d)
read -r -d '' SCRIPT << EOM\n
{training_code}
EOM
printf "%s" "$SCRIPT" > $program_path/ephemeral_script.py
read -r -d '' SCRIPT << EOM\n
{deepspeed_config}
EOM
printf "%s" "$SCRIPT" > $program_path/deepspeed.json
torchrun --nnodes $WORLD_SIZE --nproc_per_node {gpu_cnt} --master-addr $MASTER_ADDR \
--master-port $MASTER_PORT --node-rank $RANK $program_path/ephemeral_script.py \
--model_path /train/model/Meta-Llama-3.1-8B-Instruct --train_dataset_path {job_workspace}/train.jsonl \
--eval_dataset_path {job_workspace}/eval.jsonl --lora_r 8 --lora_alpha 16 --lora_dropout 0.05 \
--training_job_name {job_name} --training_job_id {job_id} --output_dir {job_workspace} \
--per_device_train_batch_size {batch_size} --per_device_eval_batch_size {batch_size} --gradient_accumulation_steps 4 \
--num_train_epochs {num_train_epochs} --eval_strategy epoch --logging_steps 1 --log_level info  \
--report_to tensorboard,mlflow --warmup_ratio 0.1 --learning_rate 2e-4 --lr_scheduler_type cosine --bf16 \
--deepspeed $program_path/deepspeed.json --gradient_checkpointing --gradient_checkpointing_kwargs {{"use_reentrant":"false"}} \
--max_seq_length 1024 --packing --eval_on_start --num_train_epoch {num_train_epochs} --disable_tqdm true
    """

    master_exec_script = exec_script.format(
        training_code=training_code, 
        deepspeed_config=deepspeed_config,
        gpu_cnt=master_resources["nvidia.com/gpu"],
        job_name=job_name, job_id=job_id,
        job_workspace=job_workspace,
        batch_size=batch_size,
        num_train_epochs=num_train_epochs
    )
    worker_exec_script = exec_script.format(
        training_code=training_code, 
        deepspeed_config=deepspeed_config,
        gpu_cnt=worker_resources["nvidia.com/gpu"]
        job_name=job_name, job_id=job_id,
        job_workspace=job_workspace,
        batch_size=batch_size,
        num_train_epochs=num_train_epochs
    )
    
    ## declare pod spec
    from kubernetes import client
    from kubeflow.training.constants import constants
    pod_template_spec = client.V1PodTemplateSpec(
        metadata=client.V1ObjectMeta(annotations={constants.ISTIO_SIDECAR_INJECTION: "false"}), ## istio sidecar disable
        spec=client.V1PodSpec(
            restart_policy="Never",
            containers=[
                client.V1Container(
                    name=constants.PYTORCHJOB_CONTAINER,
                    image=base_image,
                    command=["bash", "-c"],
                    args=[master_exec_script],
                    resources=client.V1ResourceRequirements(
                        limits=master_resources,
                        requests=master_resources
                    ),
                    env=[ ## for mlflow artifact store
                        client.V1EnvVar(
                            name="AWS_ACCESS_KEY_ID",
                            value_from=client.V1EnvVarSource(
                                secret_key_ref=client.V1SecretKeySelector(
                                    name="mlpipeline-minio-artifact",
                                    key="accesskey",
                                )
                            )
                        ),
                        client.V1EnvVar(
                            name="AWS_SECRET_ACCESS_KEY",
                            value_from=client.V1EnvVarSource(
                                secret_key_ref=client.V1SecretKeySelector(
                                    name="mlpipeline-minio-artifact",
                                    key="secretkey",
                                )
                            )
                        ),
                    ],
                    volume_mounts=[
                        client.V1VolumeMount(
                            mount_path=mount_path,
                            name=pvc_name,
                            read_only=False,

                        ),
                        client.V1VolumeMount(
                            mount_path="/dev/shm",
                            name="dshm",
                            read_only=False,
                        ),
                    ]
                )
            ],
            volumes=[
                client.V1Volume(
                    name=pvc_name,
                    persistent_volume_claim=client.V1PersistentVolumeClaimVolumeSource(
                        claim_name=pvc_name,
                        read_only=False,
                    )
                ),
                client.V1Volume(
                    name="dshm",
                    empty_dir=client.V1EmptyDirVolumeSource(
                        medium="Memory",
                        size_limit="1.0Gi"
                    )
                )
            ]
        )        
    )
    
    ## get namespace
    with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace", "r") as f:
        namespace = f.read()
    
    ## declare pytorchjob
    from kubeflow.training import models
    
    pytorchjob_name = f"{job_name}-{job_id}"
    pytorchjob = models.KubeflowOrgV1PyTorchJob(
        api_version=f"{constants.KUBEFLOW_GROUP}/{constants.OPERATOR_VERSION}",
        kind=constants.PYTORCHJOB_KIND,
        metadata=client.V1ObjectMeta(name=pytorchjob_name, namespace=namespace),
        spec=models.KubeflowOrgV1PyTorchJobSpec(
            run_policy=models.KubeflowOrgV1RunPolicy(clean_pod_policy=None),
            pytorch_replica_specs={}
        )
    )
    pytorchjob.spec.pytorch_replica_specs[constants.REPLICA_TYPE_MASTER] = models.KubeflowOrgV1ReplicaSpec(replicas=master_replica, template=pod_template_spec)
    
    import copy
    worker_pod_template_spec = copy.deepcopy(pod_template_spec)
    worker_pod_template_spec.spec.containers[0].args = [worker_exec_script]
    worker_pod_template_spec.spec.containers[0].resources = client.V1ResourceRequirements(
        limits=worker_resources,
        requests=worker_resources,
    )
    pytorchjob.spec.pytorch_replica_specs[constants.REPLICA_TYPE_WORKER] = models.KubeflowOrgV1ReplicaSpec(replicas=worker_replica, template=worker_pod_template_spec)
        
    
    ## create pytorchjob
    from kubeflow.training import TrainingClient
    training_client = TrainingClient()
    training_client.create_pytorchjob(pytorchjob, namespace)
    
    ## wait till Running state
    running_pytorchjob = training_client.wait_for_job_conditions(
        name=pytorchjob.metadata.name,
        namespace=pytorchjob.metadata.namespace,
        job_kind=constants.PYTORCHJOB_KIND,
        expected_conditions={constants.JOB_CONDITION_RUNNING}
    )
    
    ## log master pod
    training_client.get_job_logs(
        name=running_pytorchjob.metadata.name,
        namespace= running_pytorchjob.metadata.namespace,
        container=constants.PYTORCHJOB_CONTAINER,
        follow=True
    )
    
    ## check job is succeeded
    training_client.wait_for_job_conditions(
        name=running_pytorchjob.metadata.name,
        namespace=running_pytorchjob.metadata.namespace,
        job_kind=constants.PYTORCHJOB_KIND,
        expected_conditions={constants.JOB_CONDITION_SUCCEEDED}
    )
    return pytorchjob_name

## pipeline

In [67]:
from kfp import dsl
from kfp import kubernetes

@dsl.pipeline(name="newjeans-fine-tuning")
def fine_tuning_pipeline(
    data_url:str="https://docs.google.com/uc?export=download&id=1ycN8UktwSiMJ0cWwPXeLVIHJpBnUgEtE&confirm=t",
    system_prompt:str="당신은 K-pop 아이돌 그룹 뉴진스(NewJeans)의 정보를 알려주는 멋진 AI 어시스턴트입니다. 모든 대화는 한국어(Korean)로 합니다.",
    base_image:str="miroirs/transformers-pytorch-deepspeed-latest-gpu:deepspeed-0.14.0-all",
    model_id:str="unsloth/llama-3-8b-Instruct",
    output_model_prefix:str="newjeans-finetuning",
    batch_size:int=4,
    num_train_epochs:int=40,
    master_replica:int=1,
    master_cpu:str="2",
    master_memory:str="55Gi",
    master_gpu:int=1,
    worker_replica:int=1,
    worker_cpu:str="2",
    worker_memory:str="20Gi",
    worker_gpu:int=1,
):
    pvc = kubernetes.CreatePVC(
        # can also use pvc_name instead of pvc_name_suffix to use a pre-existing PVC
        pvc_name='newjeans-finetuning-pvc-rwx',
        access_modes=['ReadWriteMany'],
        size='1024Gi',
        storage_class_name='filestore-rwx',
    )
    pvc.set_caching_options(True)
    
    mount_path = '/data'
    
    dataset = txt_to_qa_dataset(data_url=data_url)
    
    chat_template_dataset = process_chat_template(
        system_prompt=system_prompt, 
        volume_mount=mount_path, 
        qa_dataset=dataset.outputs["qa_dataset"],
    )
    kubernetes.mount_pvc(chat_template_dataset, pvc_name=pvc.outputs['name'], mount_path=mount_path)
    
    train_func = save_train_func(volume_mount=mount_path)
    kubernetes.mount_pvc(train_func, pvc_name=pvc.outputs['name'], mount_path=mount_path)
    
    deepspeed_config = save_deepspeed_config(volume_mount=mount_path)
    kubernetes.mount_pvc(deepspeed_config, pvc_name=pvc.outputs['name'], mount_path=mount_path)
    
    pytorchjob = run_pytorchjob(
        base_image=base_image,
        pvc_name=pvc.outputs['name'],
        train_func=train_func.output,
        pretrained_model_name=model_id,
        dataset_name=chat_template_dataset.output,
        output_model=output_model_prefix,
        deepspeed_config_file=deepspeed_config.output,
        batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        master_replica=master_replica,
        master_cpu=master_cpu,
        master_memory=master_memory,
        master_gpu=master_gpu,
        worker_replica=worker_replica,
        worker_cpu=worker_cpu,
        worker_memory=worker_memory,
        worker_gpu=worker_gpu,
    )
    kubernetes.mount_pvc(pytorchjob, pvc_name=pvc.outputs['name'], mount_path=mount_path)

In [68]:
from kfp.compiler import Compiler

Compiler().compile(fine_tuning_pipeline, package_path="newjeans-fine-tuning.yaml")

# Serving pipeline

In [5]:
from kfp import dsl

@dsl.component(base_image="python:3.11", packages_to_install=["kserve"])
def run_isvc(
    pvc_name: str,
    pytorchjob_name: str,
):
    from kserve import constants
    from kserve import (
        V1beta1PredictorSpec,
        V1beta1InferenceServiceSpec,
        V1beta1InferenceService,
    )
    from kubernetes import client

    ## get namespace
    with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace", "r") as f:
        namespace = f.read()
    predictor_spec = V1beta1PredictorSpec(
        containers=[
            client.V1Container(
                args=[
                    "--model-id",
                    f"/mnt/models/{pytorchjob_name}",
                    "--quantize",
                    "bitsandbytes-nf4",
                ],
                image="ghcr.io/huggingface/text-generation-inference:2.0",
                name="kserve-container",
                ports=[
                    client.V1ContainerPort(
                        container_port=8080,
                        protocol="TCP",
                    )
                ],
                resources=client.V1ResourceRequirements(
                    limits={"cpu": "2", "memory": "40Gi", "nvidia.com/gpu": "1"},
                    requests={"cpu": "2", "memory": "25Gi", "nvidia.com/gpu": "1"},
                ),
                volume_mounts=[
                    client.V1VolumeMount(
                        mount_path="/mnt/models",
                        name="models",
                        read_only=False,
                    ),
                    client.V1VolumeMount(
                        mount_path="/dev/shm",
                        name="dshm",
                        read_only=False,
                    ),
                ],
            )
        ],
        max_replicas=1,
        min_replicas=1,
        volumes=[
            client.V1Volume(
                name="models",
                persistent_volume_claim=client.V1PersistentVolumeClaimVolumeSource(
                    claim_name=pvc_name,
                    read_only=False,
                ),
            ),
            client.V1Volume(
                name="dshm",
                empty_dir=client.V1EmptyDirVolumeSource(
                    medium="Memory", size_limit="0.5Gi"
                ),
            ),
        ],
    )
    
    inference_service_spec = V1beta1InferenceServiceSpec(predictor=predictor_spec)
    inference_service = V1beta1InferenceService(
        api_version=constants.KSERVE_V1BETA1,
        kind=constants.KSERVE_KIND,
        metadata=client.V1ObjectMeta(
            name="newjeans", 
            namespace=namespace, 
            annotations={
                "sidecar.istio.io/inject": "false",
                "serving.kserve.io/enable-prometheus-scraping": "true"
            }),
        spec=inference_service_spec,
    )

    from kserve import KServeClient
    kserve_client = KServeClient()
    kserve_client.create(inference_service, namespace=namespace)

## pipeline

In [ ]:
from kfp import dsl
from kfp import kubernetes

@dsl.pipeline(name="newjeans-fine-tuning")
def fine_tuning_pipeline(
    data_url:str="https://docs.google.com/uc?export=download&id=1ycN8UktwSiMJ0cWwPXeLVIHJpBnUgEtE&confirm=t",
    system_prompt:str="당신은 K-pop 아이돌 그룹 뉴진스(NewJeans)의 정보를 알려주는 멋진 AI 어시스턴트입니다. 모든 대화는 한국어(Korean)로 합니다.",
    base_image:str="miroirs/transformers-pytorch-deepspeed-latest-gpu:deepspeed-0.14.0-all",
    model_id:str="unsloth/llama-3-8b-Instruct",
    output_model_prefix:str="newjeans-finetuning",
    batch_size:int=4,
    num_train_epochs:int=40,
    master_replica:int=1,
    master_cpu:str="2",
    master_memory:str="55Gi",
    master_gpu:int=1,
    worker_replica:int=1,
    worker_cpu:str="2",
    worker_memory:str="20Gi",
    worker_gpu:int=1,
):
    pvc = kubernetes.CreatePVC(
        # can also use pvc_name instead of pvc_name_suffix to use a pre-existing PVC
        pvc_name='newjeans-finetuning-pvc-rwx',
        access_modes=['ReadWriteMany'],
        size='1024Gi',
        storage_class_name='filestore-rwx',
    )
    pvc.set_caching_options(True)
    
    mount_path = '/data'
    
    dataset = txt_to_qa_dataset(data_url=data_url)
    
    chat_template_dataset = process_chat_template(
        system_prompt=system_prompt, 
        volume_mount=mount_path, 
        qa_dataset=dataset.outputs["qa_dataset"],
    )
    kubernetes.mount_pvc(chat_template_dataset, pvc_name=pvc.outputs['name'], mount_path=mount_path)
    
    train_func = save_train_func(volume_mount=mount_path)
    kubernetes.mount_pvc(train_func, pvc_name=pvc.outputs['name'], mount_path=mount_path)
    
    deepspeed_config = save_deepspeed_config(volume_mount=mount_path)
    kubernetes.mount_pvc(deepspeed_config, pvc_name=pvc.outputs['name'], mount_path=mount_path)
    
    pytorchjob = run_pytorchjob(
        base_image=base_image,
        pvc_name=pvc.outputs['name'],
        train_func=train_func.output,
        pretrained_model_name=model_id,
        dataset_name=chat_template_dataset.output,
        output_model=output_model_prefix,
        deepspeed_config_file=deepspeed_config.output,
        batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        master_replica=master_replica,
        master_cpu=master_cpu,
        master_memory=master_memory,
        master_gpu=master_gpu,
        worker_replica=worker_replica,
        worker_cpu=worker_cpu,
        worker_memory=worker_memory,
        worker_gpu=worker_gpu,
    )
    kubernetes.mount_pvc(pytorchjob, pvc_name=pvc.outputs['name'], mount_path=mount_path)

    isvc = run_isvc(
        pvc_name=pvc.outputs["name"],
        pytorchjob_name=pytorchjob.output,
    )

In [ ]:
from kfp.compiler import Compiler

Compiler().compile(fine_tuning_pipeline, package_path="newjeans-fine-tuning.yaml")